In [8]:
%pip uninstall -y scikit-learn scipy numpy
%pip cache purge
%pip install "numpy<2.0.0" scipy scikit-learn

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Note: you may need to restart the kernel to use updated packages.
Files removed: 4
Note: you may need to restart the kernel to use updated packages.
     |████████████████████████████████| 20.6 MB 767 kB/s eta 0:00:01
     |████████████████████████████████| 39.4 MB 846 kB/s eta 0:00:01     |█████████████████████████▎      | 31.1 MB 416 kB/s eta 0:00:20
     |████████████████████████████████| 12.1 MB 558 kB/s eta 0:00:01
You should consider upgrading via the '/Users/ellenying/Projects/LIBS/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 确保 src/ 包可被导入
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, root_dir)

from config import COAL_TYPES, TRAIN_DIR, TEST_DIR
from src.data   import load_labels, load_coal_spectra
from src.model  import train_coal_model, predict_coal
from src.submit import pack_submission

In [2]:
label_map, aux_map = load_labels()

In [3]:
type(label_map), type(aux_map)

(dict, dict)

In [4]:
label_map[('煤场混煤', '1月1日')]

4439.0

In [5]:
aux_map[('煤场混煤', '1月1日')]

{'全水分': 9.6, '灰分': 36.78, '氢': 1.94, '硫': 0.34}

In [6]:
# construct label_list and aux_list from label_map and aux_map
keys = list(label_map.keys())
label_list = np.array([label_map[k] for k in keys])
aux_list = np.zeros((len(keys), len(aux_map[keys[0]].items())), dtype=np.float32)

for i, k in enumerate(keys):
    aux_list[i] = np.array(list(aux_map[k].values()), dtype=np.float32)

In [7]:
# constructing groups from the keys
# inverse mapping for coal types for easier group assignment
coal_type_inverse_map = {v: i for i, v in enumerate(COAL_TYPES)}
groups = np.array([coal_type_inverse_map[k[0]] for k in keys])

In [8]:
from config import ALPHAS
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

# 1. Scale the features properly
scaler = StandardScaler()
X_aux_scaled = scaler.fit_transform(aux_list)

In [9]:
# 2. Use GroupKFold to mimic the competition structure
# (Assuming you have access to the 'groups' array from data loading)
gkf = GroupKFold(n_splits=5)
splits = gkf.split(X_aux_scaled, label_list, groups=groups)
oof_predictions = np.zeros(len(label_list))

for train_idx, val_idx in splits:
    model = RidgeCV(alphas=ALPHAS)
    model.fit(X_aux_scaled[train_idx], label_list[train_idx])
    oof_predictions[val_idx] = model.predict(X_aux_scaled[val_idx])

In [10]:
rmse = np.sqrt(np.mean((oof_predictions - label_list) ** 2))
print(f"Cross-Validated RMSE: {rmse:.4f}")
print(f"Mean of labels: {np.mean(label_list):.4f}, Std of labels: {np.std(label_list):.4f}")
print(f"Percentage error of prediction: {rmse / np.mean(label_list) * 100:.2f}%")

Cross-Validated RMSE: 70.4117
Mean of labels: 4183.6857, Std of labels: 765.9204
Percentage error of prediction: 1.68%


The linear model is an accurate model to predict coal heat value from its chemical compositions even if we are missing the C and O components. Let's now try if a nonlinear model would do a better job.

In [13]:
from sklearn.ensemble import HistGradientBoostingRegressor
oof_predictions_gbm = np.zeros(len(label_list))

for train_idx, val_idx in splits:
    
    # Initialize the Scikit-learn LightGBM equivalent
    # We tweak hyperparameters for our tiny 5-feature dataset
    model_gbm = HistGradientBoostingRegressor(
        max_iter=150,          # number of trees
        learning_rate=0.05,     # Step size shrinkage
        max_depth=4,            # Limit tree depth to prevent overfitting
        max_leaf_nodes=15,      # Equivalent to LightGBM's num_leaves
        min_samples_leaf=10,    # Prevent isolated, overfitted samples in leaves
        random_state=42
    )

    model_gbm.fit(aux_list[train_idx], label_list[train_idx])
    oof_predictions_gbm[val_idx] = model_gbm.predict(aux_list[val_idx])


In [14]:
rmse_lgbm = np.sqrt(np.mean((oof_predictions_gbm - label_list) ** 2))
print(f"Cross-Validated RMSE (LightGBM): {rmse_lgbm:.4f}")
print(f"Percentage error of prediction (LightGBM): {rmse_lgbm / np.mean(label_list) * 100:.2f}%")

Cross-Validated RMSE (LightGBM): 4253.2176
Percentage error of prediction (LightGBM): 101.66%


This is not an improvement. The relationship between chemical components and the heat values is likely to be linear.

In [ ]:
# The following code gives a baseline RMSE using RidgeCV without cross-validation
# # which is not recommended for model evaluation. It's commented out to avoid confusion.
# model = RidgeCV(alphas=ALPHAS)
# model.fit(aux_list, label_list)
# predictions = model.predict(aux_list)
# rmse = np.sqrt(np.mean((predictions - label_list) ** 2))
# print(f"RMSE: {rmse:.4f}") 